# Build a Lighter MedBot with TinyLlama


# 1. Install Deps

In [1]:
!pip -q install -U transformers datasets accelerate peft bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 61.0 MB/s eta 0:00:00


## 2. Imports and setup

In [2]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 3. Configuration


In [3]:
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME = "medalpaca/medical_meadow_wikidoc"
OUTPUT_DIR = "tinyllama_medbot_adapter"

MAX_LENGTH = 256
TRAIN_SAMPLES = 2000
EVAL_SAMPLES = 200
BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_STEPS = 120
LEARNING_RATE = 2e-4

## 4. Load tokenizer and quantized base model


In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print("Base model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Base model loaded.


## 5. Add LoRA adapters

LoRA keeps most of the original model frozen and only trains a small number of added weights.  
That makes fine-tuning much lighter and faster.

In [5]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    pct = 100 * trainable_params / all_params
    print(f"Trainable params: {trainable_params:,}")
    print(f"All params: {all_params:,}")
    print(f"Trainable%: {pct:.4f}%")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
print_trainable_parameters(model)

Trainable params: 12,615,680
All params: 628,221,952
Trainable%: 2.0082%


## 6. Load and prepare the medical dataset


In [6]:
raw_ds = load_dataset(DATASET_NAME)
raw_ds

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc.json:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 10000
    })
})

In [7]:
def format_example(example):
    instruction = example.get("instruction", "") or ""
    input_text = example.get("input", "") or ""
    output_text = example.get("output", "") or ""

    if input_text.strip():
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output_text}"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n{output_text}"
    return {"text": prompt}

formatted_ds = raw_ds.map(format_example)

train_ds = formatted_ds["train"].shuffle(seed=42).select(range(min(TRAIN_SAMPLES, len(formatted_ds["train"]))))
eval_ds = formatted_ds["train"].shuffle(seed=7).select(
    range(min(EVAL_SAMPLES, len(formatted_ds["train"])))
)

print("Train rows:", len(train_ds))
print("Eval rows:", len(eval_ds))
print(train_ds[0]["text"][:700])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Train rows: 2000
Eval rows: 200
### Instruction:
Answer this question truthfully

### Input:
How does Nitrendipine work?

### Response:
Once Nitrendipine is ingested, it is absorbed by the gut and metabolized by the liver before it goes into the systemic circulation and reaches the cells of the smooth muscles and cardiac muscle cells. It binds more effectively with L-type calcium channels in smooth muscle cells because of its lower resting membrane potential.  The Nitrendipine diffuses into the membrane and binds to its high affinity binding site on the inactivated L-type calcium channel that’s located in between each of the 4 intermembrane components of the α1 subunit . The exact mechanism of action of Nitrendipine is unk


## 7. Tokenize the dataset


In [8]:
def tokenize_function(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = train_ds.map(tokenize_function, batched=True, remove_columns=train_ds.column_names)
tokenized_eval = eval_ds.map(tokenize_function, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
tokenized_train, tokenized_eval

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 2000
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 200
 }))

## 8. Training arguments


In [9]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    warmup_steps=10,
    logging_steps=10,
    save_steps=40,
    eval_strategy="steps",
    eval_steps=40,
    fp16=torch.cuda.is_available(),
    report_to="none",
    optim="paged_adamw_8bit",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

trainer

## 9. Train the model

In [10]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
40,1.652835,1.608542
80,1.598534,1.572308
120,1.593731,1.560004


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=120, training_loss=1.6420334259668985, metrics={'train_runtime': 63.9714, 'train_samples_per_second': 30.013, 'train_steps_per_second': 1.876, 'total_flos': 3088106316103680.0, 'train_loss': 1.6420334259668985, 'epoch': 0.96})

## 10. Save the adapter and tokenizer

This saves the LoRA adapter weights, configuration, and tokenizer files.  
These are the files you will later load for inference or deployment.

In [11]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved files:")
print(os.listdir(OUTPUT_DIR))

Saved files:
['checkpoint-120', 'README.md', 'checkpoint-80', 'tokenizer_config.json', 'chat_template.jinja', 'adapter_config.json', 'adapter_model.safetensors', 'checkpoint-40', 'tokenizer.json']


## 11. Reload for inference

For inference, we:
1. reload the quantized base model
2. load the saved LoRA adapter on top
3. generate a response to a medical question

In [12]:
inference_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
inference_model = PeftModel.from_pretrained(inference_model, OUTPUT_DIR)
inference_model.eval()

print("Inference model ready.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inference model ready.


## 12. Test the MedBot

In [13]:
def generate_medbot_response(question, max_new_tokens=120):
    prompt = f"### Instruction:\nAnswer the following medical question clearly and briefly.\n\n### Input:\n{question}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

question = "What are the common symptoms of diabetes?"
response = generate_medbot_response(question)
print(response)

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Answer the following medical question clearly and briefly.

### Input:
What are the common symptoms of diabetes?

### Response:
Common symptoms of diabetes include:
Diabetes Mellitus Type 1 Diabetes Mellitus Type 2 Diabetes Mellitus Type 3 Other complications
Symptoms of diabetes include:
Severe thirst Dry mouth Dry eyes Dry mouth with dry mouth 
Symptoms of diabetes include:
Severe thirst Dry mouth Dry eyes Dry mouth with dry mouth
Symptoms of diabetes include:
Severe thirst Dry mouth D


# 13. Download Model

In [15]:
import os
import shutil

# Remove checkpoint folders
for item in os.listdir(OUTPUT_DIR):
    item_path = os.path.join(OUTPUT_DIR, item)
    if os.path.isdir(item_path) and item.startswith("checkpoint"):
        shutil.rmtree(item_path)

print("Cleaned directory:")
print(os.listdir(OUTPUT_DIR))

Cleaned directory:
['README.md', 'tokenizer_config.json', 'chat_template.jinja', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json']


In [16]:
from google.colab import files

zip_path = shutil.make_archive(f"/content/{OUTPUT_DIR}", "zip", OUTPUT_DIR)
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>